##### Project name: Teva GPT - Genie Supervisor
##### Project owner, Team: Anat Sadeh, DSET
##### Notebook Author, Team: Jeharul Shaik, DSET
##### Date: 30/01/2026
##### Purpose of Notebook: To create MCP servers for all the Genie Spaces across the Databricks workspaces and assign an LLM to act as a Supervisor. When a user queries in natural language, the Supervisor decides which dataset(s) added under a specific Genie space to redirect the question to.

##### Notebook Process Flow:
1. First, the agent.py file gets created which will have access to all the Genie Spaces (in the form of tools)
2. Log the agent created above as an MLflow model
3. Register the model to Unity Catalog
2. Create a serving endpoint using the model from the Unity Catalog and pass the Key vault secrets from the advanced properties section

#### Mosaic AI Agent Framework: Author and deploy a Genie MCP tool - calling LangGraph agent

This notebook shows how to author a LangGraph agent that connects to Genie MCP servers hosted on Databricks. You can choose between a Databricks-managed MCP server, a custom MCP server hosted as a Databricks app, or both simultaneously. To learn more about these options, see [MCP on Databricks](https://docs.databricks.com/aws/en/generative-ai/mcp/).


This notebook uses the [`ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) interface for compatibility with Mosaic AI features. In this notebook you learn to:

- Author a LangGraph agent (wrapped with `ResponsesAgent`) that calls Genie MCP tools
- Manually test the agent
- Log and deploy the agent into the Unity Catalog

To learn more about authoring an agent using Mosaic AI Agent Framework, see Databricks documentation ([Azure](https://learn.microsoft.com/azure/databricks/generative-ai/agent-framework/create-chat-model)).

In [0]:
# Install and upgrade required Python libraries directly into the notebook environment.
# %pip is the preferred way to manage Python dependencies inside Databricks notebooks.
# Packages:
# - langgraph: Framework for building stateful AI agents and graphs
# - uv: Extremely fast Python package installer and runtime environment manager
# - databricks-agents: Databricks toolkit for building and running AI agents
# - mlflow-skinny[databricks]: Lightweight MLflow build with Databricks-specific extras
# - databricks-mcp: Databricks integration for MCP (Model Context Protocol) tools
# - databricks-langchain: LangChain extensions optimized for Databricks and Unity Catalog
# - mcp==1.22.0: Pin a specific MCP core library version for compatibility/stability
# -qqqq: Makes the notebook execution cleaner by eliminating almost all pip logs.

%pip install -U -qqqq \
    langgraph \
    uv \
    databricks-agents \
    mlflow-skinny[databricks] \
    databricks-mcp \
    databricks-langchain \
    mcp==1.22.0

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart the Python interpreter for this Databricks notebook session.
# This is required after installing packages with %pip install so that:
#  - Newly installed or upgraded libraries become available
#  - Old cached modules are removed
#  - Dependency conflicts are resolved
# CAUTION: This clears all variables and state from the current notebook.
dbutils.library.restartPython()

<B> Environment variable(s) creation in the Notebook & Serving Endpoint </B> 

We are setting the key vault secrets into the environment variable so that the agent runs without any failure locally (<B>dbutils</B> is not accessible in the agent.py).

In [0]:
# Read the Key vault secrets and set them into the necessary environment variables. "dbutils" utility is only available inside the Databricks Notebooks and not inside the normal python file(s) (i.e. agaent.py).

import os
# os.environ["AZURE_CLIENT_ID"] = dbutils.secrets.get(scope="<COMMENTING_FOR_SECURITY_REASONS>", key="<COMMENTING_FOR_SECURITY_REASONS>")  
# os.environ["AZURE_CLIENT_SECRET"] = dbutils.secrets.get(scope="<COMMENTING_FOR_SECURITY_REASONS>", key="<COMMENTING_FOR_SECURITY_REASONS>")

### Define the agent code

Define the agent code in a single cell below. This lets you easily write the agent code to a local Python file, using the `%%writefile` magic command, for subsequent logging and deployment.

The following cell creates a flexible, tool-using agent that integrates Databricks Genie MCP servers with the Mosaic AI Agent Framework. Here’s what happens, at a high level:

1. **MCP tool wrappers**  
   Custom wrapper classes are defined so LangChain tools can interact with Databricks Genie MCP servers. You can connect to Databricks-managed MCP servers, custom MCP servers hosted as a Databricks App, or both.

2. **Tool discovery & registration**  
   The agent automatically discovers available tools from the specified Genie MCP server(s), turns their metadata into Python objects, and prepares them for the LLM.

3. **Define the LangGraph agent logic**  
   Define an agent workflow that does the following:
   - The agent reads messages (conversation/history).
   - If a tool (Genie Space) call is requested, it’s executed using the correct Genie MCP tool.
   - The agent can loop, performing multiple tool calls as needed, until a final answer is ready.

4. **Wrap the LangGraph agent using the `ResponsesAgent` class**  
   The agent is wrapped using `ResponsesAgent` so it's compatible with the Mosaic AI.
   
5. **MLflow autotracing**
   Enable MLflow autologging to start automatic tracing.

In [0]:
%%writefile agent.py
import requests  # HTTP client library for calling REST APIs
import asyncio  # Provides async/await support for asynchronous programming
from typing import Annotated, Any, Generator, List, Optional, Sequence, TypedDict, Union  # Type hints for better code clarity and tooling

import mlflow  # ML lifecycle management: tracking, models, and deployment
import nest_asyncio  # Allows nested event loops, useful in notebooks running asyncio
from databricks.sdk import WorkspaceClient  # Databricks REST API client for workspace operations
from databricks_langchain import (  # LangChain integrations optimized for Databricks
    ChatDatabricks,                   # LangChain LLM wrapper for Databricks model serving
    UCFunctionToolkit,                # Lets LangChain call Unity Catalog Python functions as tools
    VectorSearchRetrieverTool,        # Tool to retrieve chunks from Databricks Vector Search
)
from databricks_mcp import DatabricksMCPClient, DatabricksOAuthClientProvider  # MCP client + OAuth provider for Databricks Model Context Protocol
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage  # Message object types used in LangChain conversations
from langchain_core.language_models import LanguageModelLike  # Common interface for LLMs in LangChain
from langchain_core.runnables import RunnableConfig, RunnableLambda  # Runnable components for building LLM pipelines
from langchain_core.tools import BaseTool  # Base class for defining tools the LLM can call
from langgraph.graph import END, StateGraph  # LangGraph primitives for building stateful agent graphs
from langgraph.graph.message import add_messages  # Helper for merging graph messages
from langgraph.prebuilt.tool_node import ToolNode  # Node wrapper allowing agents to call tools
from mcp import ClientSession  # MCP session manager for client interactions
from mcp.client.streamable_http import streamablehttp_client as connect  # Streaming HTTP MCP client connector
from mlflow.pyfunc import ResponsesAgent  # MLflow wrapper for serving agents as pyfunc models
from mlflow.types.responses import (  # MLflow response message/types for agent interfaces
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from pydantic import create_model  # Dynamically creates validated data models
# from pyspark.sql import SparkSession  # Spark entry point (commented out)
import os, json, time  # OS utilities, JSON handling, and timing functions
from msal import ConfidentialClientApplication  # Microsoft identity library for Azure AD OAuth auth flows
from typing import Dict, List  # Additional typing aliases
nest_asyncio.apply()  # Enable nested event loops so asyncio works inside notebooks

# def get_dbutils():
#     spark = SparkSession.builder.getOrCreate()
#     try:
#         from pyspark.dbutils import DBUtils
#         return DBUtils(spark)
#     except ImportError:
#         import IPython
#         return IPython.get_ipython().user_ns["dbutils"]

# dbutils = get_dbutils()

# CLIENT_ID = os.environ["AZURE_CLIENT_ID"]
# CLIENT_SECRET = os.environ["AZURE_CLIENT_SECRET"]

# LIST OF WORKSPACES: List of Databricks workspace URLs to process
"""
We are adding the list of Databriks workspaces to a python list and not inside a seperate config file. This is due to the unavailability of config file's path during the servind endpoint creation process.
"""
WORKSPACES = [
    "https://adb-984752964297111.11.azuredatabricks.net/", # RND Dev Workspace
    # "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net", # NAC Dev Workspace
    # "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net", #  TGO Dev Workspace
    # "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net", # DSET Dev Workspace
    # "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net", # IM Dev Workspace
    # "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net" # EUC Dev Workspace
  ]
# Azure AD Tenant ID, used for authentication with Microsoft Entra ID (AAD)
# TENANT_ID = "<COMMENTING_FOR_SECURITY_REASONS>" 
 # DATABRICKS SCOPE: The scope for which the AAD token is requested (usually an App ID URI)
# SCOPE = "<COMMENTING_FOR_SECURITY_REASONS>"  
# RESULTS: List to store details of Genie spaces and their permissions for each workspace
RESULTS = []


# -----------------------------
# 1) Acquire Azure AD Token
# -----------------------------

"""
This function uses the MSAL library to acquire an Azure AD token for the service principal.
It authenticates using the service principal credentials and requests a token for the specified scope.
Returns the access token string if successful, otherwise raises an Exception.

Example input:
    (no arguments, however it uses the variables which are mentioned above -> CLIENT_ID, CLIENT_SECRET, TENANT_ID, SCOPE)
Example output:
    'eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIn...'
"""
# def get_aad_token():
#     try:
#         # Create a ConfidentialClientApplication instance for service principal authentication
#         app = ConfidentialClientApplication(
#             client_id=CLIENT_ID,
#             authority=f"https://login.microsoftonline.com/{TENANT_ID}",
#             client_credential=CLIENT_SECRET,
#         )
#         # Acquire token for the specified scope (resource)
#         token = app.acquire_token_for_client(scopes=[SCOPE])
#         # Check if access_token is present in the response
#         if "access_token" not in token:
#             raise Exception("Failed to get token")
#         return token["access_token"]
#     except Exception as error:
#         # Log and re-raise any exception encountered during token acquisition
#         print(f"Error acquiring Azure AD token: {error}")
#         #send_logs_eventhub("Error", "get_aad_token", str(error))
#         raise

# -----------------------------
# Helpers
# -----------------------------
"""
Helper functions to create a Databricks PAT, fetch Genie spaces, and fetch Genie space permissions.
Each function includes detailed comments and error handling.
"""
# def create_pat(workspace_url, aad_token_value):
#     """
#     Create a Databricks Personal Access Token (PAT) for a given workspace using an Azure AD token. Note: This PAT token is created dynamically and is not stored / hardcoded anywhere.
#     Args:
#         workspace_url (str): The URL of the Databricks workspace.
#         aad_token_value (str): The Azure AD access token.
#     Returns:
#         str or None: The PAT token value if successful, None otherwise. 

#     Example input:
#         workspace_url = "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net"
#         aad_token_value = "eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIn..."
#     Example output:
#         "dapikey1234567890abcdef"
#     """
#     try:
#         # Construct the API endpoint for creating a PAT
#         url = f"{workspace_url}/api/2.0/token/create"
#         # Send POST request to create a PAT with a 1-hour lifetime
#         response = requests.post(
#             url,
#             json={"comment": "sp", "lifetime_seconds": 3600},
#             headers={"Authorization": f"Bearer {aad_token_value}"},
#             timeout=8,
#         )
#         # Check if the response status code indicates success (200 or 201)
#         if response.status_code not in (200, 201):
#             print(f"Failed to create PAT for {workspace_url}: {response.status_code} {response.text}")
#             #send_logs_eventhub("Error", "create_pat", f"{response.status_code} {response.text}")
#             return None
#         # Return the PAT token value from the response
#         return response.json().get("token_value")
#     except Exception as error:
#         # Log any exception and return None to indicate failure
#         print(f"Exception in create_pat for {workspace_url}: {error}")
#         #send_logs_eventhub("Error", "create_pat", str(error))
#         return None
    

# Fetch Genie spaces for a given Databricks workspace using a PAT.
def get_spaces(workspace_url, pat_token):
    """
    Fetch Genie spaces for a given Databricks workspace using a PAT.
    Args:
        workspace_url (str): The URL of the Databricks workspace.
        pat_token (str): The Databricks PAT token.
    Returns:
        list: List of Genie space objects (dicts), or empty list if error.

    Example input:
        workspace_url = "https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net"
        pat_token = "dapikey1234567890abcdef"
    Example output:
        [
            {"space_id": "abc123", "title": "Finance"},
            {"space_id": "def456", "title": "R&D"}
        ]
    """
    try:
        # Construct the API endpoint for fetching Genie spaces
        url = f"{workspace_url}/api/2.0/genie/spaces"
        # Send GET request to fetch Genie spaces
        response = requests.get(
            url, headers={"Authorization": f"Bearer {pat_token}"}
        )
        # Check if the response status code indicates success
        if response.status_code != 200:
            print(f"Failed to fetch Genie spaces for {workspace_url}: {response.status_code} {response.text}")
            #send_logs_eventhub("Error", "get_spaces", f"{response.status_code} {response.text}")
            return []
        # Return the list of Genie spaces from the response
        return response.json().get("spaces", [])
    except Exception as error:
        # Log any exception and return empty list
        print(f"Exception in get_spaces for {workspace_url}: {error}")
        #send_logs_eventhub("Error", "get_spaces", str(error))
        return []
    

############################################
## Define your LLM endpoint
############################################
# This is an LLM serving endpoint that the Databricks provide. We need this to decide which Genie space(Genie MCP Tool) should be selected based on the User's question.
LLM_ENDPOINT_NAME = "databricks-claude-sonnet-4-6"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME) # ChatDatabricks is the official LangChain chat model class for calling Databricks‑hosted LLMs (Foundation Model APIs, provisioned serving endpoints, Genie models, etc.) directly from LangChain.

system_prompt = None # Setting the system prompt to None as the same is being sent from the application calling the Serving Endpoint. This is done to have more control over the prompt while changing the inputs dynamically on a need basis.

#####################
## MCP Tool Creation
#####################

# Define a custom LangChain tool that wraps functionality for calling MCP servers
class MCPTool(BaseTool):
    """
    A custom LangChain Tool that acts as a thin wrapper around a Databricks
    MCP (Model Context Protocol) server tool. You register it with an agent
    so the LLM can decide when to call the underlying MCP tool.

    Why a wrapper? LangChain Tools provide a standard, typed interface that
    agents understand; this class adapts MCP tool invocation to that interface.

    BaseTool is the abstract base class that all LangChain tools inherit from.
    Tools are actions an agent can take in our case, invoking an MCP tool (Genie Spaces).
    LangChain agents rely on Tools to perform external operations that LLMs cannot do themselves.
    """

    def __init__(
        self,
        name: str,
        description: str,
        args_schema: type,
        server_url: str,
        ws: WorkspaceClient,
        is_custom: bool = False,
    ):
        # Initialize Tool base fields that LangChain agents rely on (name, description, args)
        # so the planner/LLM can discover the tool and validate inputs.
        super().__init__(name=name, description=description, args_schema=args_schema)

        """
        Why we use `object.__setattr__` instead of normal assignment (`self.x = y`)
        -------------------------------------------------------------------------

        LangChain's `BaseTool` is *not* a normal Python class. It behaves like a 
        Pydantic-style validated data model with a "frozen" structure:

        • Every field on a Tool must be declared in advance (name, description,
            args_schema, return_direct, etc.).
        • BaseTool overrides `__setattr__` to *validate or restrict* attribute
            assignment so that unexpected fields cannot be added accidentally.
        • This makes Tool objects behave like structured, immutable specifications,
            which agents rely on for safety and consistency.

        Because of this, writing:
            self.server_url = server_url
            self.workspace_client = ws
            self.is_custom = is_custom

        will NOT work. BaseTool intercepts these through its overridden __setattr__ and
        rejects them because they are *not declared fields* in the Tool schema.

        To safely store internal implementation details (like MCP server URLs,
        workspace clients, or toggles), we must **bypass BaseTool’s setter logic**.

        Python gives us exactly one correct way to do this:

        object.__setattr__(self, "server_url", server_url)

        Using `object.__setattr__`:
        ✓ Directly writes into the instance dictionary
        ✓ Skips BaseTool’s overridden __setattr__
        ✓ Avoids validation or immutability restrictions
        ✓ Prevents recursive setter calls
        ✓ Allows us to attach internal state without affecting the Tool’s public schema

        In summary:
        -----------
        `BaseTool` enforces schema-based immutability → normal assignment fails.  
        `object.__setattr__` bypasses those restrictions → internal fields are safely stored.  
        This is the recommended approach when extending Tool behavior while keeping the
        Tool schema clean, predictable, and agent-friendly.
        """
        object.__setattr__(self, "server_url", server_url)
        object.__setattr__(self, "workspace_client", ws)
        object.__setattr__(self, "is_custom", is_custom)


    def _run(self, **kwargs) -> str:
        """
        Synchronous execution path required by LangChain Tool interface.

        If this tool targets a custom MCP server, we defer to the async implementation
        (wrapping it in asyncio.run). Otherwise, we use the Databricks managed MCP
        client in a straightforward synchronous call.

        Why two paths?
        - Managed MCP servers on Databricks expose ready-to-use endpoints and are
          commonly called via a convenience client synchronously.
        - Custom servers typically use an explicit async ClientSession + OAuth
        """
        if self.is_custom:
            """
            When Custome MCP servers (not provided by Databricks) are being used.
            This is not necessary for Databricks managed MCP servers (we are not using it)
            """
            return asyncio.run(self._run_custom_async(**kwargs))
        else:
            """
             Why this is needed:
             -------------------
             Databricks MCP servers (Genie Spaces in this case) expose
             a standardized interface where AI agents call "tools" dynamically. 
             
             The `DatabricksMCPClient` handles:
               • authenticating with the workspace (via WorkspaceClient)
               • connecting to the MCP server endpoint
               • listing/invoking available tools dynamically
             
             According to Databricks MCP documentation, MCP follows a client–server model
             where the client requests tool execution and the server returns a structured
             response. The agent should NOT hardcode tool behavior, but call them through
             the MCP client exactly like done here.
            """
            mcp_client = DatabricksMCPClient(
                server_url=self.server_url,          # The MCP server endpoint (e.g., Genie Space MCP)
                workspace_client=self.workspace_client  # Auth + permissions enforced via Unity Catalog by using the WorkspaceClient which is part of the Databricks SDK for Python, and it exposes every workspace‑level REST API in one unified object.
            )

            """
             Call the MCP tool exposed under `self.name`.
            
             What `.call_tool()` does:
             -------------------------
             • Sends a tool invocation request to the chosen MCP server.
             • MCP servers return a "content array" — a list of structured content parts,
               usually containing text chunks. This is part of the MCP protocol. 
            
             Databricks explicitly advises that:
               - Tools can evolve over time,
               - Output formats are NOT guaranteed to stay fixed,
               - And the *LLM* should interpret the output instead of writing rigid parsers.
             
             This call therefore returns a response whose `.content` is a list such as:
               [
                   ContentPart(text="first chunk of data"),
                   ContentPart(text="more details..."),
                   ...
               ]
            """
            response = mcp_client.call_tool(self.name, kwargs)

            """
             MCP tool output flattening:
             ---------------------------
             Because MCP returns a list of content parts, the simplest and safest approach
             (recommended by Databricks) is to join the text segments and let the LLM 
             interpret them. We intentionally avoid complex parsing — per Databricks
             guidelines — because MCP tool outputs may evolve and the agent should handle
             them generatively.
            
             Example:
             --------
             Suppose the MCP tool returns:
               response.content = [
                   ContentPart(text="User has 12 active records."),
                   ContentPart(text=" Last updated 2 hours ago.")
               ]
            
             This join produces:
               "User has 12 active records. Last updated 2 hours ago."
            
             The agent/LLM then reasons over the combined string.
            """
            return "".join([content.text for content in response.content])

    async def _run_custom_async(self, **kwargs) -> str:
        """
        Asynchronous execution path for custom MCP servers.
        We are not using this at the moment as we do not have any custom MCPs to use.
        """
        async with connect(
            self.server_url,
            auth=DatabricksOAuthClientProvider(self.workspace_client)
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            # Create an MCP client session on the transport and initialize the protocol.
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                # Call the MCP tool exposed by the custom server with the provided kwargs.
                response = await session.call_tool(self.name, kwargs)

                # Return concatenated text content (again: keep parsing minimal and let the LLM handle semantics). [2](https://docs.databricks.com/aws/en/generative-ai/mcp/)
                return "".join([content.text for content in response.content])


# Retrieve tool definitions from a managed MCP server.
def get_managed_mcp_tools(ws: WorkspaceClient, server_url: str):
    
    """ 
    This method gets tool definitions from a managed MCP server and returns a list of Tool objects

    The DatabricksMCPClient handles:
               • authenticating with the workspace (via WorkspaceClient)
               • connecting to the MCP server endpoint
               • listing/invoking available tools dynamically

    Sampe Input:
    ------------
    server_url: https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net/api/2.0/mcp/genie/01f0f799bb041e18aa98537c93881301
    ws: WorkspaceClient(host='https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net', auth_type='pat', ...)

    Sample Output:
    --------------           
    
    [Tool(name='query_space_01f0f799bb041e18aa98537c93881301', title=None, description='Query the RND_GMA_Genie_Demo genie space for data insights.\nYou can ask natural language questions and will receive responses in natural language or as SQL query results.\nYou can ask for a summary of the datasets in the genie space to get an overview of the data available.\nBy default, each query is standalone. Optionally, provide a conversation_id to continue an existing conversation.\nIf you do not have a conversation_id, please provide all relevant context in the query.\nThe response will include the conversation_id, message_id, and status of the message in the genie space.\nIf the message is not complete, use poll_response_01f0f799bb041e18aa98537c93881301 with the returned conversation_id and message_id to poll until the message reaches a completed state.\nThe genie space description is as follows:\n', inputSchema={'type': 'object', 'required': ['query'], 'properties': {'query': {'type': 'string', 'description': 'Query for genie space'}, 'conversation_id': {'type': 'string', 'description': 'Optional conversation ID to continue an existing conversation'}}}, outputSchema={'type': 'object', 'required': ['content', 'conversationId', 'messageId', 'status'], 'properties': {'content': {'type': 'object', 'properties': {'queryAttachments': {'type': 'array', 'items': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'description': {'type': 'string'}, 'statement_response': {}}}}, 'textAttachments': {'type': 'array', 'items': {'type': 'string'}}}}, 'conversationId': {'type': 'string'}, 'messageId': {'type': 'string'}, 'status': {'type': 'string'}}}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None), Tool(name='poll_response_01f0f799bb041e18aa98537c93881301', title=None, description='Poll for the response of a previously initiated message in the RND_GMA_Genie_Demo genie space.\nUse this tool to retrieve results for a message that was started but not yet completed.\nRequires both conversation_id and message_id returned from the initial query_space_01f0f799bb041e18aa98537c93881301 tool call.\nPlease continue polling with this tool until the message reaches a completed state.', inputSchema={'type': 'object', 'required': ['conversation_id', 'message_id'], 'properties': {'conversation_id': {'type': 'string', 'description': 'The conversation ID from the genie space'}, 'message_id': {'type': 'string', 'description': 'The message ID to poll for the response of'}}}, outputSchema={'type': 'object', 'required': ['content', 'conversationId', 'messageId', 'status'], 'properties': {'content': {'type': 'object', 'properties': {'queryAttachments': {'type': 'array', 'items': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'description': {'type': 'string'}, 'statement_response': {}}}}, 'textAttachments': {'type': 'array', 'items': {'type': 'string'}}}}, 'conversationId': {'type': 'string'}, 'messageId': {'type': 'string'}, 'status': {'type': 'string'}}}, icons=None, annotations=ToolAnnotations(title=None, readOnlyHint=True, destructiveHint=None, idempotentHint=None, openWorldHint=None), meta=None)]
    """

    mcp_client = DatabricksMCPClient(server_url=server_url, workspace_client=ws)
    return mcp_client.list_tools()


# Convert an MCP tool definition into a LangChain-compatible tool
def create_langchain_tool_from_mcp(
    mcp_tool, server_url: str, ws: WorkspaceClient, is_custom: bool = False
):
    """
        Create a LangChain tool from an MCP tool definition

        Sampe Input:
        ------------
        mcp_tool : The output of the get_managed_mcp_tools as shown above
        server_url: https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net/api/2.0/mcp/genie/01f0f799bb041e18aa98537c93881301
        ws: WorkspaceClient(host='https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net', auth_type='pat', ...)

        Sample Output:
        --------------
        name='query_space_01f0f799bb041e18aa98537c93881301' description='Query the RND_GMA_Genie_Demo genie space for data insights.\nYou can ask natural language questions and will receive responses in natural language or as SQL query results.\nYou can ask for a summary of the datasets in the genie space to get an overview of the data available.\nBy default, each query is standalone. Optionally, provide a conversation_id to continue an existing conversation.\nIf you do not have a conversation_id, please provide all relevant context in the query.\nThe response will include the conversation_id, message_id, and status of the message in the genie space.\nIf the message is not complete, use poll_response_01f0f799bb041e18aa98537c93881301 with the returned conversation_id and message_id to poll until the message reaches a completed state.\nThe genie space description is as follows:\n' args_schema=<class 'agent.query_space_01f0f799bb041e18aa98537c93881301Args'> 
    """

    schema = mcp_tool.inputSchema.copy()
    properties = schema.get("properties", {})
    required = schema.get("required", [])

    # Map JSON schema types to Python types for input validation
    TYPE_MAPPING = {"integer": int, "number": float, "boolean": bool}
    field_definitions = {}
    for field_name, field_info in properties.items():
        field_type_str = field_info.get("type", "string")
        field_type = TYPE_MAPPING.get(field_type_str, str)

        if field_name in required:
            field_definitions[field_name] = (field_type, ...)
        else:
            field_definitions[field_name] = (field_type, None)

    """
        Dynamically create a Pydantic schema for the tool's input arguments
        
        Examples: <For each genie space MCP tool>
        ----------
        <class 'agent.query_space_01f0f799bb041e18aa98537c93881301Args'>
        <class 'agent.poll_response_01f0f799bb041e18aa98537c93881301Args'>
    """ 
    args_schema = create_model(f"{mcp_tool.name}Args", **field_definitions)

    # Return a configured MCPTool instance
    return MCPTool(
        name=mcp_tool.name,
        description=mcp_tool.description or f"Tool: {mcp_tool.name}",
        args_schema=args_schema,
        server_url=server_url,
        ws=ws,
        is_custom=is_custom,
    )


# Gather all tools from managed and custom MCP servers into a single list
async def create_mcp_tools_genie() -> List[MCPTool]:
    """
        Create LangChain tools from both managed and custom MCP servers
        
        Sample Output: List of all the tools from the MCP Servers
        --------------
        [name='query_space_01f0f799bb041e18aa98537c93881301' description='Query the RND_GMA_Genie_Demo genie space for data insights.\nYou can ask natural language questions and will receive responses in natural language or as SQL query results.\nYou can ask for a summary of the datasets in the genie space to get an overview of the data available.\nBy default, each query is standalone. Optionally, provide a conversation_id to continue an existing conversation.\nIf you do not have a conversation_id, please provide all relevant context in the query.\nThe response will include the conversation_id, message_id, and status of the message in the genie space.\nIf the message is not complete, use poll_response_01f0f799bb041e18aa98537c93881301 with the returned conversation_id and message_id to poll until the message reaches a completed state.\nThe genie space description is as follows:\n' args_schema=<class 'agent.query_space_01f0f799bb041e18aa98537c93881301Args'>,
        
        name='poll_response_01f0f799bb041e18aa98537c93881301' description='Poll for the response of a previously initiated message in the RND_GMA_Genie_Demo genie space.\nUse this tool to retrieve results for a message that was started but not yet completed.\nRequires both conversation_id and message_id returned from the initial query_space_01f0f799bb041e18aa98537c93881301 tool call.\nPlease continue polling with this tool until the message reaches a completed state.' args_schema=<class 'agent.poll_response_01f0f799bb041e18aa98537c93881301Args'>]

    """
    tools = []
    genie_mcp_list = []

    # Get the Azure AD token for authenticating to Databricks REST API
    # aad_token = get_aad_token()

    for workspace_url in WORKSPACES:
        try:
            # For each workspace, create a PAT using the Azure AD token
            # pat_token = create_pat(workspace_url, aad_token)
            pat_token = dbutils.secrets.get(scope="<COMMENTING_FOR_SECURITY_REASONS>", key="<COMMENTING_FOR_SECURITY_REASONS>")
            if not pat_token:
                # If PAT creation failed, append an error entry to RESULTS and continue to next workspace
                RESULTS.append(
                    {"workspace": workspace_url, "error": "pat_failed"}
                )
                continue

            # Extract the domain name of the databricks workspace alone
            # For example: From the following URL https://<COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net, 
            # only extract <COMMENTING_FOR_SECURITY_REASONS>.azuredatabricks.net
            workspace_client = WorkspaceClient(host=workspace_url.replace("https://", "").split("/")[0],token=pat_token)

             # Fetch Genie spaces for the workspace using the PAT
            for genie_space in get_spaces(workspace_url, pat_token):
                genie_mcp_list.append(workspace_url+"/api/2.0/mcp/genie/"+genie_space.get("space_id"))
            print ("Genie mcp list {}".format(genie_mcp_list))

            for server_url in genie_mcp_list:
                try:
                    # Retrieve tool definitions from a managed MCP server.
                    mcp_tools = get_managed_mcp_tools(workspace_client, server_url)
                    for mcp_tool in mcp_tools:
                        # Convert an MCP tool definition into a LangChain-compatible tool
                        tool = create_langchain_tool_from_mcp(mcp_tool, server_url, workspace_client, is_custom=False)
                        tools.append(tool)
                except Exception as err:
                    print(f"Error loading tools from managed server {server_url}: {err}")
        except Exception as error:
            # Log any exception encountered while processing a workspace
            print(f"Exception processing workspace {workspace_url}: {error}")
            #send_logs_eventhub("Error", "process_workspace", str(error))
            RESULTS.append(
                {"workspace": workspace_url, "error": str(error)}
            )      
      
    return tools


#####################
## Define agent logic
#####################

# The state for the agent workflow, including the conversation and any custom data
class AgentState(TypedDict):
    messages: Annotated[Sequence[AnyMessage], add_messages]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


# The state for the agent workflow, including the conversation and any custom data
def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    system_prompt: Optional[str] = None,
):
    """
    Build a minimal, controllable *tool-calling agent* using LangGraph + LangChain.
    
    • We "bind" tool definitions to the chat model so it can issue structured tool calls.
    • We define a tiny state machine with two nodes:
          (1) the 'agent' node → runs the LLM,
          (2) the 'tools' node → executes any tool calls requested by the LLM.
    • We loop: agent → (if tool_calls) tools → agent … until no more tool calls.
    • We optionally prepend a system prompt as the first message each time we call the model.
    """

    # -------------------------------
    # 1) Bind tools to the LLM model
    # -------------------------------
    # `bind_tools` attaches tool schemas to the model so it can *decide* to call them.
    # After binding, the model may produce an AIMessage with .tool_calls populated.
    model = model.bind_tools(tools)  # Bind tools to the model

    # --------------------------------------------------------------
    # 2) Router function: should we keep going, or finish the graph?
    # --------------------------------------------------------------
    # LangGraph uses *conditional edges*. We supply a function that looks at the current
    # state and returns a string key to decide the next hop.
    # Rule here:
    #   - If the *last LLM message* includes tool calls, go to the "tools" node ("continue").
    #   - Else, stop execution ("end").
    def should_continue(state: AgentState):
        messages = state["messages"]          # conversation so far (list[BaseMessage])
        last_message = messages[-1]           # LLM just responded, so check that
        # If function (tool) calls are present, continue; otherwise, end
        if isinstance(last_message, AIMessage) and last_message.tool_calls:
            return "continue"
        else:
            return "end"

    # -------------------------------------------------------------------------
    # 3) Optional preprocessor: prepend a system prompt to the message history
    # -------------------------------------------------------------------------
    # RunnableLambda is a tiny, composable step in the LangChain Expression Language (LCEL).
    # We use it to transform incoming state → a message list for the model.
    if system_prompt:
        # If a system prompt is provided, every model invocation sees it as the first message.
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}] + state["messages"]
        )
    else:
        # No system prompt? Just pass the messages straight through.
        preprocessor = RunnableLambda(lambda state: state["messages"])

    # --------------------------------------------------------
    # 4) Compose a runnable pipeline: preprocessor → model
    # --------------------------------------------------------
    # The `|` operator chains runnables: output of left feeds into right.
    # When we invoke `model_runnable`, it will:
    #   a) build the message list (with or without system prompt),
    #   b) call the bound model to get an AIMessage (possibly with tool_calls).
    model_runnable = preprocessor | model  # Chain the preprocessor and the model

    # -----------------------------------------------------------
    # 5) Define the function the 'agent' node will actually run
    # -----------------------------------------------------------
    # LangGraph node functions receive the current `state` and (optionally) a `config`.
    # We call the model and *append* its response into the state under "messages".
    def call_model(
        state: AgentState,
        config: RunnableConfig,
    ):
        # model_runnable.invoke(...) expects the *whole state* because the preprocessor
        # accesses state["messages"]. It returns an AIMessage.
        response = model_runnable.invoke(state, config)
        # Node functions return a *partial state update*; LangGraph merges it into global state.
        return {"messages": [response]}

    # ---------------------------------------------------
    # 6) Build the LangGraph state machine for the agent
    # ---------------------------------------------------
    workflow = StateGraph(AgentState)  # Create the agent's state machine, typed on AgentState

    # Add the two nodes:
    #   • "agent" → runs the LLM (wrapped as a RunnableLambda so graph can call it)
    #   • "tools" → runs the tools the model requested (ToolNode knows how to execute them)
    workflow.add_node("agent", RunnableLambda(call_model))  # Agent node (LLM)
    workflow.add_node("tools", ToolNode(tools))             # Tools node

    # Start at the agent node
    workflow.set_entry_point("agent")  # Start at agent node

    # When "agent" finishes, evaluate `should_continue(state)`:
    #   - if it returns "continue", route to "tools"
    #   - if it returns "end", stop the graph
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",  # If the model requests a tool call, move to tools node
            "end": END,           # Otherwise, end the workflow
        },
    )

    # After the "tools" node executes the requested tool(s), send control back to the agent
    # so the model can read tool results and decide whether to call another tool or finish.
    workflow.add_edge("tools", "agent")  # After tools are called, return to agent node

    # ---------------------------------------------------------
    # 7) Compile the graph → a runnable app with .invoke/.stream
    # ---------------------------------------------------------
    # `compile()` statically validates the graph and returns a runnable object
    # you can call with an initial state, e.g., {"messages": [HumanMessage(...)]}.
    return workflow.compile()



# This class adapts a LangGraph-compiled agent (self.agent) to a simple
# "ResponsesAgent" interface with both batch (`predict`) and streaming
# (`predict_stream`) methods.
#
# - self.agent is a LangGraph app that exposes `.stream(...)`, which yields a
#   sequence of events during execution (node updates and message chunks).
# - We translate those LangGraph events into your "ResponsesAgent" protocol:
#   * predict(...)        -> returns a ResponsesAgentResponse
#   * predict_stream(...) -> yields ResponsesAgentStreamEvent as the agent works
# - ResponsesAgent               : protocol/base class your apps consume
# - ResponsesAgentRequest        : input payload (messages + custom inputs)
# - ResponsesAgentResponse       : final output container (items + custom outputs)
# - ResponsesAgentStreamEvent    : per-chunk/per-item streaming event
# - to_chat_completions_input    : helper that converts your request messages into
#                                  the model’s expected "chat completion" format
# - output_to_responses_items_stream : helper that converts LangChain messages
#                                      into your ResponsesAgent stream items
# - AIMessageChunk               : streaming chunk type produced by LangChain models

class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, agent):
        # Keep a reference to the compiled LangGraph application.
        # This object must expose `.stream(input_state, stream_mode=...)`
        # and typically `.invoke(input_state)` if you also support non-streaming.
        self.agent = agent

    # ---------------------------
    # High-level, single-call API
    # ---------------------------
    # predict() is a convenience wrapper around predict_stream():
    #   1) Run the streaming generator to completion
    #   2) Collect final output items and errors
    #   3) Return a ResponsesAgentResponse
    #
    # This is useful for synchronous web handlers or tests where you want a
    # single call that blocks until the agent is done.
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        # Drain the stream and keep only "output item is done" or "error" events.
        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done" or event.type == "error"
        ]

        # Pass through any caller-provided "custom_inputs" into custom_outputs
        # so the consumer can correlate request/response if needed.
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    # -----------------------------------------
    # Low-level, incremental streaming interface
    # -----------------------------------------
    # predict_stream() yields events as they are produced:
    #   • node "updates" (intermediate tool/LLM messages)
    #   • "messages" (token / chunk deltas for live text)
    #
    # This makes it easy to wire to a WebSocket/SSE endpoint or a CLI progress UI.
    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:

        # 1) Convert the input messages into the "chat completion" shape that
        #    your model / LangGraph expects. We call .model_dump() on each input
        #    (e.g., Pydantic model) then pass the list to a helper that produces
        #    the standard roles/contents structure.
        cc_msgs = to_chat_completions_input([input.model_dump() for input in request.input])

        # 2) Stream from the LangGraph application.
        #    self.agent.stream(...) yields a sequence of 2-tuples:
        #      ("updates", <dict_of_node_partial_states>)  OR
        #      ("messages", <list_of_message_chunks>)
        #
        #    stream_mode=["updates", "messages"] asks LangGraph to push both:
        #      • "updates"  -> when a node (e.g., tools/agent) updates its state
        #      • "messages" -> token/segment chunks from the LLM (AIMessageChunk)
        for event in self.agent.stream({"messages": cc_msgs}, stream_mode=["updates", "messages"]):

            if event[0] == "updates":
                # event[1] is a dict keyed by node name; values are partial state updates.
                # We want to push any incremental "messages" found in each node’s update.
                for node_data in event[1].values():
                    # If the node posted new messages, convert them into your streaming items
                    # and yield one-by-one. The helper encapsulates parsing of LangChain
                    # message types (HumanMessage, AIMessage, ToolMessage, etc.).
                    if len(node_data.get("messages", [])) > 0:
                        # yield from allows the helper to produce 0..N events seamlessly
                        yield from output_to_responses_items_stream(node_data["messages"])

            elif event[0] == "messages":
                # event[1] is typically a list of chunks (e.g., streaming tokens/segments)
                # from the LLM. We only care about AIMessageChunk with non-empty content.
                try:
                    chunk = event[1][0]
                    if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                        # Build a delta event that carries the incremental text,
                        # plus an item_id so the client can group deltas.
                        yield ResponsesAgentStreamEvent(
                            **self.create_text_delta(delta=content, item_id=chunk.id),
                        )
                except:
                   pass


# Initialize the entire agent, including MCP tools and workflow
def initialize_agent():
    """
    Initialize the agent with MCP tools sourced from Genie Spaces.

    High-level flow:
      1) Build a set of MCP tools from your configured Genie Spaces
         (managed MCP servers inside the Databricks workspace).
      2) Assemble a tool-calling agent (LLM + tools + optional system prompt).
      3) Wrap that agent in a LangGraph-compatible ResponsesAgent so you can
         run it with durable, stateful graph semantics
    """
    
    # Build the set of tools exposed by our Genie Space MCP server.
    # - `create_mcp_tools_genie()` is an *async* function (it likely connects to your MCP
    #   server, discovers the available tools, and returns them as a Python list).
    # - Because this function is async, we cannot call it directly from normal (sync) code.
    #   `asyncio.run(...)` creates a temporary event loop, runs the coroutine to completion,
    #   and returns its result.
    # - End result: `mcp_tools` becomes a list of tool objects the agent can bind to and call.

    mcp_tools = asyncio.run(
        create_mcp_tools_genie()
    )

    # Create the agent graph with an LLM, tool set, and system prompt (if given)
    # Inputs referenced below:
    #   • `llm`           : a chat model client (Claude Serving Endpoint).
    #   • `mcp_tools`     : the MCP-backed tools returned above (Genie).
    #   • `system_prompt` : optional string with system guidance (may be None).

    agent = create_tool_calling_agent(llm, mcp_tools, system_prompt)

    
    # ---------------------------------------------------------------------
    # Wrap the agent with a LangGraph-compatible facade
    # ---------------------------------------------------------------------
    # Returning `LangGraphResponsesAgent` gives you graph execution semantics:
    #   • Durable state (checkpointing, resume, audit).
    #   • Cycles/branches (supervisor-worker patterns, HITL pauses).
    #   • A simple `.invoke(...)` / `.predict(...)` style surface for apps.
    #
    # In practice, this means you get a “graph runner” around your agent so
    # you can integrate it into pipelines, Apps, and production workflows.

    return LangGraphResponsesAgent(agent)


mlflow.langchain.autolog()
AGENT = initialize_agent()
mlflow.models.set_model(AGENT)

Overwriting agent.py


## Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

In [0]:
# Import the initialized agent instance from the agent.py module
# AGENT is a LangGraphResponsesAgent that wraps a compiled LangGraph workflow
# with MCP tools (Genie Spaces) bound to a Claude LLM endpoint
from agent import AGENT

# Invoke the agent's predict method to get a complete response (non-streaming)
# What happens internally:
# 1. The input is converted to chat completion format via to_chat_completions_input()
# 2. The LangGraph agent workflow starts at the "agent" node (LLM)
# 3. The LLM (Claude) analyzes the query and decides which MCP tool to call
# 4. If tool calls are needed, the workflow routes to the "tools" node
# 5. The appropriate Genie Space MCP tool (query_space_<space_id>) is invoked
# 6. Genie processes the natural language query against its configured datasets
# 7. Tool results are returned to the LLM for interpretation
# 8. The LLM may call additional tools (like poll_response) if needed
# 9. The workflow loops (agent → tools → agent) until no more tool calls
# 10. Final response is returned as a ResponsesAgentResponse object

# Sample Input : {"input": [{"role": "user", "content": "How many dummy datasets do we have"}]}
# Sample Output: ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='message', id='lc_run--019c23d0-b2c0-72f0-809b-5e9f30da5978', content=[{'text': "I'll query the Dummy Genie Demo space to find out how many dummy datasets are available.", 'type': 'output_text'}], role='assistant'), OutputItem(type='function_call', id='lc_run--019c23d0-b2c0-72f0-809b-5e9f30da5978', call_id='toolu_bdrk_015boi2FdJChHuuRmdcKihdr', name='query_space_01ef87ab51f41d4cbb7ea2f93ebf6a19', arguments='{"query": "How many dummy datasets do we have?"}'), OutputItem(type='function_call_output', call_id='toolu_bdrk_015boi2FdJChHuuRmdcKihdr', output='{"content":{"queryAttachments":[],"textAttachments":["There are three dummy datasets available: 10000dummy_data, 200000dummy_data, and 2000000dummy_data."]},"conversationId":"01f101091e8f12a28e813ef476b1fab8","messageId":"01f101091e9f1489a2d6efa2cf355dce","status":"COMPLETED"}'), OutputItem(type='message', id='lc_run--019c23d0-eebb-7112-9996-f21d08592774', content=[{'text': 'Based on the query results, **we have 3 dummy datasets**:\n\n1. 10000dummy_data\n2. 200000dummy_data\n3. 2000000dummy_data\n\nThese datasets appear to be named based on their size (10,000, 200,000, and 2,000,000 records respectively).', 'type': 'output_text'}], role='assistant')], parallel_tool_calls=None, temperature=None, tools=None, top_p=None, max_output_tokens=None, previous_response_id=None, reasoning=None, status=None, text=None, usage=None, user=None, custom_outputs=None)

AGENT.predict({"input": [{"role": "user", "content": "How many dummy datasets do we have in Phaxton Pharma"}]})

Genie mcp list ['https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126c2f03c1b77ab2e9838246d4762', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126c2694618bbaee9dfb3d2c91f77', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126c2524c1fe6a1b9854cf10e91b6', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126c10c861aad994a5375eaee51d9', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126c04bc11b4a8083a4cfc5f9358e', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126b586f812e0aafe8f0f8c4b603c', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126ae34371e99be6ee23fa8b3c02d', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126a97c471d38b381263291cf2e96', 'https://adb-984752964297111.11.azuredatabricks.net//api/2.0/mcp/genie/01f126a3339e184c972487c3cc7cab08', 'https://adb-984752964297111.1

ResponsesAgentResponse(tool_choice=None, truncation=None, id=None, created_at=None, error=None, incomplete_details=None, instructions=None, metadata=None, model=None, object='response', output=[OutputItem(type='message', id='lc_run--019d1b37-8359-7603-a09f-bedd1f11e105', content=[{'text': 'I don\'t have a specific Genie Space for "Phaxton Pharma" in my available tools. However, let me search across the available spaces to see if any of them might contain relevant information. Let me check a few spaces that could potentially have this data.\n\nThat said, none of the Genie Spaces I have access to appear to be specifically related to **Phaxton Pharma**. The available spaces I can query are:\n\n1. **Drop Table Demo (1 & 2)** – Testing table recreation behavior\n2. **Marketing Analytics Assistant** – Google Ads, TikTok Ads campaign performance\n3. **NHLBI Grants and Funding Analytics** – NIH grant funding data\n4. **Finance and Vendor Analytics** – Financial and vendor data\n5. **Order Metr

Trace(trace_id=tr-9610b00a5f6c901a9c39e7b9c0996037)

In [0]:

# Navigate: AGENT -> compiled LangGraph -> PregelNode -> ToolNode
tool_node = AGENT.agent.nodes["tools"].bound
tool_map = tool_node.tools_by_name

print(f"Total tools available: {len(tool_map)}\n")
for name, tool in tool_map.items():
    desc_preview = (tool.description[:150].replace('\n', ' ') + '...') if tool.description else 'N/A'
    print(f"  Tool: {name}")
    print(f"  Description: {desc_preview}")
    print(f"  Args schema: {tool.args_schema.schema() if tool.args_schema else 'N/A'}")
    print()

Total tools available: 40

  Tool: query_space_01f126c2f03c1b77ab2e9838246d4762
  Description: Query the Drop Table Demo (2) genie space for data insights. You can ask natural language questions and will receive responses in natural language or ...
  Args schema: {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'conversation_id': {'default': None, 'title': 'Conversation Id', 'type': 'string'}}, 'required': ['query'], 'title': 'query_space_01f126c2f03c1b77ab2e9838246d4762Args', 'type': 'object'}

  Tool: poll_response_01f126c2f03c1b77ab2e9838246d4762
  Description: Poll for the response of a previously initiated message in the Drop Table Demo (2) genie space. Use this tool to retrieve results for a message that w...
  Args schema: {'properties': {'conversation_id': {'title': 'Conversation Id', 'type': 'string'}, 'message_id': {'title': 'Message Id', 'type': 'string'}}, 'required': ['conversation_id', 'message_id'], 'title': 'poll_response_01f126c2f03c1b77ab2e9838246d4762Ar

/home/spark-11cdc95e-a8b4-4d85-8152-f4/.ipykernel/24198/command-7662611755703811-2856811042:11: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(f"  Args schema: {tool.args_schema.schema() if tool.args_schema else 'N/A'}")


## Log the agent as an MLflow model

Log the agent as code from the `agent.py` file. See [Deploy an agent that connects to Databricks MCP servers](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp#deploy-your-agent).

In [0]:

# Import MLflow so we can log the agent as an MLflow model.
# MLflow acts as Databricks’ standard packaging format for deploying
# agents, LLM pipelines, and functions to Model Serving / AI Gateway.
import mlflow

# This constant holds the name of a Databricks Foundation Model Serving endpoint
# (like “databricks-meta-llama-3-1-70b-instruct”) that the agent will call.
# The agent.py file should reference this to make LLM calls inside the model.
from agent import LLM_ENDPOINT_NAME

# MLflow "resources" allow your model to declare *external dependencies*
# required at serving time.
# DatabricksServingEndpoint → makes the agent depend on an LLM endpoint.
# DatabricksFunction → makes the agent depend on a UC Python function.
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction

# get_distribution() lets us dynamically capture the installed version of packages.
# This ensures we lock agent dependencies to EXACT versions when logging the model.
from pkg_resources import get_distribution


# ------------------------------------------------------------------------------------
# Resources the agent needs at SERVING TIME.
# These are NOT pip packages — they are Databricks-managed external compute resources.
# ------------------------------------------------------------------------------------
resources = [
    # This tells MLflow: “When this agent runs, it needs access to LLM endpoint”.
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),

    # This registers a Unity Catalog Python function the agent may call
    # via Databricks Functions / SCP (system.ai.python_exec).
    DatabricksFunction(function_name="system.ai.python_exec")
]


# ------------------------------------------------------------------------------------
# Start an MLflow run.
# Everything logged inside is grouped under one run in the MLflow UI.
# ------------------------------------------------------------------------------------
with mlflow.start_run():

    # Log our agent as an MLflow pyfunc model.
    #
    # Key parameters:
    # ----------------------------
    # name="agent"
    #   → The name assigned to the model in the MLflow registry.
    #
    # python_model="agent.py"
    #   → MLflow will import and run the PythonModel inside agent.py.
    #     That PythonModel must implement `predict()` or `load_context()`.
    #
    # resources=[...]
    #   → Declare all Databricks-managed external dependencies your agent needs.
    #     Model Serving will validate these and attach correct permissions.
    #
    # pip_requirements=[...]
    #   → Exact Python packages to recreate the environment for this agent
    #     at serving time (in Model Serving or AI Gateway).
    #   → We dynamically freeze versions (e.g., langgraph==<installed_version>).
    #
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,

        pip_requirements=[
            "databricks-mcp",   # Needed for MCP client/server protocol
            f"langgraph=={get_distribution('langgraph').version}",  # exact version pin
            f"mcp=={get_distribution('mcp').version}",              # exact version pin
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            "msal==1.32.3",  # Azure authentication helper for token-based access (if used)
        ]
    )


## Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
import mlflow

# Tell MLflow that when we refer to a model "registry",
# we want to use **Unity Catalog (UC)** instead of the old MLflow workspace registry.
#
# Why this matters:
# -----------------
# • "databricks-uc" switches MLflow to the UC Model Registry.
# • UC Model Registry stores models in a governed catalog/schema.
# • This enables:
#     - Access control at the catalog/schema level
#     - Cross-workspace sharing
#     - Lineage and governance per Unity Catalog rules

mlflow.set_registry_uri("databricks-uc")

# -----------------------------------------------------------
# TODO: You define where in Unity Catalog the model should live.
# A UC model has a three-part fully qualified name:
#     <catalog>.<schema>.<model>
# -----------------------------------------------------------
catalog = "<COMMENTING_FOR_SECURITY_REASONS>"   # Example UC catalog (like "main" or "production")
schema = "<COMMENTING_FOR_SECURITY_REASONS>"     # Example schema within the catalog
model_name = "tevagpt_genie_supervisor_claude-sonnet-4-6"  # The logical MLflow model name (choose any valid identifier)

# Combine them into a UC-style fully qualified model name
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# ---------------------------------------------------------------------------
# Register the model with Unity Catalog Model Registry.
#
# What this does:
# ---------------
# • Takes the model you logged earlier (logged_agent_info.model_uri)
#   and registers it under your UC model name.
# • Creates the FIRST version of the model inside UC.
# • After registration, you will have:
#       UC path: sample_catalog.sample_schema.sample_model
#       Version: v1 (created now)
#
# • From here, you can:
#       - Deploy it to Model Serving
#       - Use it via MLflow client APIs
#       - Use UC permissions to control access
#       - Track lineage and versioning in UC

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,  
    name=UC_MODEL_NAME                      
)

## Deploy the agent
You should deploy the model by directly going to the model page from the specified Unity Catalog path and by clicking on the "Serve this model" button from the top right of the page and providing your own name for the serving endpoint.

When the same values which were set inside the environment variables in the cells above need to be pulled during the creation of Servine Endpoint, we need to pass them via the Advanced configuration section of the serving endpoint

